In [6]:
from datasets import load_dataset

data=load_dataset("glue","rte")

In [7]:
data

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 2490
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 277
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3000
    })
})

In [8]:
data["train"][0]


{'sentence1': 'No Weapons of Mass Destruction Found in Iraq Yet.',
 'sentence2': 'Weapons of Mass Destruction Found in Iraq.',
 'label': 1,
 'idx': 0}

In [1]:
from transformers import AutoTokenizer,AutoModelForSequenceClassification,TrainingArguments,Trainer



In [2]:
tokenizer=AutoTokenizer.from_pretrained("bert-base-uncased")

C:\Users\Lenovo\AppData\Roaming\Python\Python311\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [11]:
def tokenize_function(data):
    return tokenizer(data["sentence1"], data["sentence2"], truncation=True)

tokenized_data = data.map(tokenize_function, batched=True)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [12]:
label2id={x["label"]: x["label"] for x in data["train"]}
label2id

{1: 1, 0: 0}

In [13]:
label2id={"Yes" : 1, "No": 0} # for RTE dataset
id2label={ v:k for k,v in label2id.items()} # vice versa of label2id
label2id

{'Yes': 1, 'No': 0}

In [14]:
from transformers import AutoConfig

config=AutoConfig.from_pretrained("bert-base-uncased",label2id=label2id,id2label=id2label)

In [15]:
model=AutoModelForSequenceClassification.from_pretrained("bert-base-uncased",config=config)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [16]:
training_args = TrainingArguments(
    output_dir="result",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    logging_steps= 150
)

In [17]:
from sklearn.metrics import accuracy_score,f1_score

def compute_metrics(eval_pred):
    predictions,labels = eval_pred
    pred= predictions.argmax(axis=1)
    accuracy = accuracy_score(labels, pred)
    f1=f1_score(labels, pred)
    return {"accuracy": accuracy,
            "f1": f1}


In [18]:
trainer=Trainer(
    
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["validation"],
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
)

In [19]:
trainer.train()

  0%|          | 0/312 [00:00<?, ?it/s]

c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': 0.6902, 'grad_norm': 5.225875377655029, 'learning_rate': 1.0384615384615386e-05, 'epoch': 0.96}


  0%|          | 0/5 [00:00<?, ?it/s]

{'eval_loss': 0.6691861748695374, 'eval_accuracy': 0.6028880866425993, 'eval_f1': 0.5769230769230769, 'eval_runtime': 79.5814, 'eval_samples_per_second': 3.481, 'eval_steps_per_second': 0.063, 'epoch': 1.0}


...

{'loss': 0.5987, 'grad_norm': 10.583974838256836, 'learning_rate': 7.692307692307694e-07, 'epoch': 1.92}


c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  0%|          | 0/5 [00:00<?, ?it/s]

{'eval_loss': 0.6610451340675354, 'eval_accuracy': 0.6101083032490975, 'eval_f1': 0.5135135135135135, 'eval_runtime': 52.9556, 'eval_samples_per_second': 5.231, 'eval_steps_per_second': 0.094, 'epoch': 2.0}
{'train_runtime': 3455.834, 'train_samples_per_second': 1.441, 'train_steps_per_second': 0.09, 'train_loss': 0.6403106267635639, 'epoch': 2.0}


TrainOutput(global_step=312, training_loss=0.6403106267635639, metrics={'train_runtime': 3455.834, 'train_samples_per_second': 1.441, 'train_steps_per_second': 0.09, 'total_flos': 418427772449640.0, 'train_loss': 0.6403106267635639, 'epoch': 2.0})

In [25]:
model.save_pretrained("model_directory")
from transformers import BertTokenizer

tokenizer.save_pretrained("model_directory")


('model_directory\\tokenizer_config.json',
 'model_directory\\special_tokens_map.json',
 'model_directory\\vocab.txt',
 'model_directory\\added_tokens.json',
 'model_directory\\tokenizer.json')

# Making Predictions with Trained Model

Now that we have trained and saved our model, let's create functions to make predictions on new sentence pairs. (Not trained well)

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

def load_trained_model(model_path="model_directory"):
    """Load the trained model and tokenizer"""
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    return model, tokenizer

def predict_entailment(sentence1, sentence2, model, tokenizer):
    """
    Predict whether sentence1 entails sentence2
    Returns: prediction probability and label
    """
    # Tokenize the input sentences
    inputs = tokenizer(sentence1, sentence2, 
                      return_tensors="pt", 
                      truncation=True, 
                      padding=True, 
                      max_length=512)
    
    # Make prediction
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probabilities = F.softmax(logits, dim=-1)
    
    # Get prediction
    predicted_class = torch.argmax(probabilities, dim=-1).item()
    confidence = probabilities[0][predicted_class].item()
    
    # Map to labels
    id2label = {0: "No", 1: "Yes"}
    predicted_label = id2label[predicted_class]
    
    return {
        "prediction": predicted_label,
        "confidence": confidence,
        "probabilities": {
            "No": probabilities[0][0].item(),
            "Yes": probabilities[0][1].item()
        }
    }

# Load the trained model
trained_model, trained_tokenizer = load_trained_model("model_directory")

In [16]:
# Example usage: Test the prediction function
test_sentence1 = "I saw a movie."
test_sentence2 = "The movie was good"

result = predict_entailment(test_sentence1, test_sentence2, trained_model, trained_tokenizer)

print(f"Sentence 1: {test_sentence1}")
print(f"Sentence 2: {test_sentence2}")
print(f"Prediction: {result['prediction']}")
print(f"Confidence: {result['confidence']:.4f}")
print(f"Probabilities: {result['probabilities']}")

Sentence 1: I saw a movie.
Sentence 2: The movie was good
Prediction: No
Confidence: 0.6068
Probabilities: {'No': 0.6068381071090698, 'Yes': 0.3931618630886078}
